### Data Ingestion

In [1]:
### Document Structure:

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the content of the document.",
    metadata={
      "source": "example.pdf", 
      "page": 1,
      "author": "John Doe",
      "date_created": "2024-06-01"
    }
)
doc

Document(metadata={'source': 'example.pdf', 'page': 1, 'author': 'John Doe', 'date_created': '2024-06-01'}, page_content='This is the content of the document.')

In [3]:
## Creating a simple text document:
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_text = {
    "../data/text_files/sample.txt": """This is a sample text document.
It contains multiple lines of text.
This is the third line.
This is the fourth line.
This is the fifth line with extra spaces.
This is the sixth line."""
}

for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text document created at:", file_path)

Sample text document created at: ../data/text_files/sample.txt


In [5]:
### Read the file (using TextLoader):
# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader
text_loader = TextLoader("../data/text_files/sample.txt", encoding="utf-8")
documents = text_loader.load()
print("Loaded documents:", documents)

C:\Users\sobha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded documents: [Document(metadata={'source': '../data/text_files/sample.txt'}, page_content='This is a sample text document.\nIt contains multiple lines of text.\nThis is the third line.\nThis is the fourth line.\nThis is the fifth line with extra spaces.\nThis is the sixth line.')]


In [6]:
### Directory loader for txt files:
from langchain_community.document_loaders import DirectoryLoader
directory_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt", # Pattern to match files
    loader_cls= TextLoader, # Use TextLoader for .txt files
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
documents = directory_loader.load()
print("Loaded documents from directory:", documents)

100%|██████████| 2/2 [00:00<00:00, 147.76it/s]

Loaded documents from directory: [Document(metadata={'source': '..\\data\\text_files\\sample.txt'}, page_content='This is a sample text document.\nIt contains multiple lines of text.\nThis is the third line.\nThis is the fourth line.\nThis is the fifth line with extra spaces.\nThis is the sixth line.'), Document(metadata={'source': '..\\data\\text_files\\sample2.txt'}, page_content="What is Lorem Ipsum?\n\nLorem Ipsum is simply dummy text of the printing and typesetting industry. \nLorem Ipsum has been the industry's standard dummy text ever since the 1500s, \nwhen an unknown printer took a galley of type and scrambled it to make a type specimen book. \nIt has survived not only five centuries, but also the leap into electronic typesetting, \nremaining essentially unchanged. \n\nIt was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, \nand more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.")

In [7]:
### Directory loader for pdf files:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
pdf_directory_loader = DirectoryLoader(
    "../data/pdf_files", 
    glob="**/*.pdf", # Pattern to match files
    loader_cls= PyMuPDFLoader, # Use PyMuPDFLoader for .pdf files
    # loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
pdf_documents = pdf_directory_loader.load()
print("Loaded pdf documents from directory:", pdf_documents)

100%|██████████| 3/3 [00:00<00:00,  8.20it/s]

Loaded pdf documents from directory: [Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'file_path': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'format': 'PDF 1.3', 'title': 'data_centers6', 'author': 'gmarcy', 'subject': '', 'keywords': '', 'moddate': "D:20260329232124Z00'00'", 'trapped': '', 'modDate': "D:20260329232124Z00'00'", 'creationDate': "D:20260329232124Z00'00'", 'page': 0}, page_content='1 \n \n \n \n \n \n \nThe Impact of Computing Data Centres Orbiting Earth \n  \nGeoffrey W. Marcy1* \n \n1 Space Laser Awareness, 3388 Petaluma Hill Rd, Santa Rosa, CA, 95404, USA \n \n \nSubmitted to Letters of the Monthly Notices of the Royal Astronomical Society \n \nAccepted: xx     \nVersion: Mar 29, 2026 \n \n \nABSTRACT \n \nArtificial intelligence is projected to increase U.S. data-centre power demand beyond 100 gigawatt (GW) \nby 2035 an

In [8]:
type(pdf_documents[0])

### To learn about other document loaders, check out the documentation at: https://docs.langchain.com/oss/python/integrations/document_loaders

langchain_core.documents.base.Document

### Embedding and Vector Store DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    """Class to handle embedding of documents using SentenceTransformer."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.
        Args:
            model_name (str): HuggingFace model name for sentence embedding.
        """
        self.model = model_name
        # self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model}")
            self.model = SentenceTransformer(self.model)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of text strings.
        Args:
            texts (List[str]): Input texts to embed.
        Returns:
            np.ndarray: Generated embedding vectors.
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embedding.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings of shape: {embeddings.shape}")
        return embeddings

    # def get_sentence_embedding_dimension(self) -> int:
    # 	 """Get the dimension of the sentence embeddings.
    # 	 Returns:
    # 		 int: Dimension of the embedding vectors.
    # 	 """
    #     if not self.model:
    #         raise ValueError("Model not loaded. Cannot get embedding dimension.")
    #     return self.model.get_sentence_embedding_dimension()

## Initialize the embedding manager:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


C:\Users\sobha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sobha\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading w

Model loaded successfully. Embedding dimension: 384


### Vector Store

In [ ]:
class VectorStore:
    """Class to manage document embeddings in a ChromaDB vector store."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store.
        Args:
            collection_name (str): Name of the ChromaDB collection to use.
            persist_directory (str): Directory to persist the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            print("ChromaDB client initialized successfully.")

            # Get or create the collection
            self.collection = self._get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents to the vector store with their embeddings.
        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document and prepare metadata and text content
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # Generate a unique ID for each document
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)  # Convert metadata to a dictionary if it's not already
            metadata['doc_index'] = i  # Add document index to metadata
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)  # Use the prepared metadata
            # Document content
            documents_text.append(doc.page_content)  # Use the document's text content
            #Embeddings
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list

        # Add data to the collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,  # Use the prepared embeddings list
                metadatas=metadatas,  # Use the prepared metadata list
                documents=documents_text  # Use the prepared document text list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore